In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
import graph_tool.all as gt
import pickle

# DATA 1

In [2]:
def import_network_data(f):
    edges = []
    for i, line in enumerate(f):
        if i == 0:
            continue  # Skip the first line
        words = line.split()
        edges.append((words[0], words[1]))
    
    f.close()
    
    # Create an undirected graph from edge list
    G = nx.Graph()
    G.add_edges_from(edges)
    
    # Remove self-loops
    G.remove_edges_from(nx.selfloop_edges(G))
    
    # Extract the largest connected component
    largest_cc = max(nx.connected_components(G), key=len)
    G = G.subgraph(largest_cc).copy()
    
    # Relabel nodes in ascending order starting from 0
    G = nx.convert_node_labels_to_integers(G, ordering="sorted")
    
    return G

In [3]:
# Load networks

# Road
f_road=open(r"../../Data/out.subelj_euroroad_euroroad")
Road=import_network_data(f_road)

# Collab
f_collab=open(r"../../Data/out.dimacs10-netscience")
Collab = import_network_data(f_collab)

# Elegans
f_elegans=open(r"../../Data/out.dimacs10-celegans_metabolic")
Elegans = import_network_data(f_elegans)

# SIF
f_sif=open(r"../../Data/out.asoiaf")
Sif = import_network_data(f_sif)

# Emails
f_emails=open(r"../../Data/out.arenas-email")
Emails = import_network_data(f_emails)

# FB
f_fb=open(r"../../Data/out.ego-facebook")
FB = import_network_data(f_fb)


data_1 = {}
data_1["E-road"] = Road
data_1["NetSci Collab"] = Collab
data_1["C. Elegans"] = Elegans
data_1["Song of Ice and Fire"] = Sif
data_1["Emails"] = Emails
data_1["Facebook"] = FB


In [15]:
# Save the dictionary to a .pkl file
with open("Intermediate_outputs_1/data_1.pkl", "wb") as f:
    pickle.dump(data_1, f)

In [20]:
# Load networkx data
#with open("Intermediate_outputs_1/data_1.pkl", "rb") as f:
    #data_1 = pickle.load(f)

In [21]:
# create graph tool from nx
# Dictionary to store graph-tool versions of the networks
dict_gt_1 = {}
for name, nx_graph in data_1.items():
    g = gt.Graph(directed=False) 
    # Ensure vertex index matches NetworkX node labels
    g.add_edge_list([(u, v) for u, v in nx_graph.edges()]) 
    dict_gt_1[name] = g


In [22]:
# now create the states of each of them, with minimize sbm
state_1={}
sbm_1 = {}
for name, gt_graph in dict_gt_1.items():
    st=[]
    entr=[]
    for i in range(5):
        state = gt.minimize_blockmodel_dl(gt_graph,state_args=dict(deg_corr=True))
        for j in range(1000):
            state.multiflip_mcmc_sweep(beta=np.inf, niter=10)
        en=state.entropy()
        st.append(state)
        entr.append(en)
    # minimum
    min_index = entr.index(min(entr))
    # best state
    best_state = st[min_index]
    b = best_state.get_blocks()
    L = {int(v): int(b[v]) for v in gt_graph.vertices()}
    sbm_1[name] = L
    state_1[name] = best_state

In [23]:
# Save the dictionary to a .pkl file
with open("Intermediate_outputs_1/sbm_1.pkl", "wb") as f:
    pickle.dump(sbm_1, f)

# Save the dictionary to a .pkl file
with open("Intermediate_outputs_1/state_1.pkl", "wb") as f:
    pickle.dump(state_1, f)

# DATA 2

In [6]:
# Load datasets

# File containing all the edgelists
file_path = "../../Data/PPT-Ohmnet_tissues-combined.edgelist"
df = pd.read_csv(file_path, sep="\t")
# Strip spaces from column names to avoid KeyError
df.columns = df.columns.str.strip()
# Dictionary to store graphs for each tissue
tissue_graphs = {}
# Group edges by tissue and create a graph for each
for tissue, group in df.groupby("tissue"):
    G = nx.Graph()
    G.add_edges_from(zip(group["# protein1"], group["protein2"]))
    tissue_graphs[tissue] = G
    
tissue_names = list(tissue_graphs.keys())  # Extract all tissue names
#print(len(tissue_names) ) # Print the list
#print(tissue_names)

# Remove self-loops from all tissue networks
for tissue in tissue_graphs:
    tissue_graphs[tissue].remove_edges_from(nx.selfloop_edges(tissue_graphs[tissue]))

# Keep only the largest connected component for each network
for tissue in tissue_graphs:
    G = tissue_graphs[tissue]
    if not nx.is_connected(G):  # If not connected
        largest_cc = max(nx.connected_components(G), key=len)
        G = G.subgraph(largest_cc).copy()

    # Relabel nodes to ascending order of appearance (0,1,2,...)
    mapping = {old_label: new_label for new_label, old_label in enumerate(sorted(G.nodes()))}
    G = nx.relabel_nodes(G, mapping)

    # Store the updated graph
    tissue_graphs[tissue] = G

In [5]:
names = ["basophil", "brain", "cardiac_muscle", "cornea", "epidermis", "jejunum" , "locus_ceruleus" , "lung" , "natural_killer_cell" , "neuron" , "ovarian_follicle" ,"t_lymphocyte"]
data_2 = {g: tissue_graphs[g] for g in names if g in tissue_graphs}
# Save the dictionary to a .pkl file
with open("Intermediate_outputs_2/data_2.pkl", "wb") as f:
    pickle.dump(data_2, f)

In [148]:
# create graph tool from nx
# Dictionary to store graph-tool versions of the networks
dict_gt_2 = {}
for name, nx_graph in data_2.items():
    g = gt.Graph(directed=False) 
    # Ensure vertex index matches NetworkX node labels
    g.add_edge_list([(u, v) for u, v in nx_graph.edges()]) 
    dict_gt_2[name] = g

In [149]:
# now create the states of each of them, with minimize sbm
state_2={}
sbm_2 = {}
for name, gt_graph in dict_gt_2.items():
    st=[]
    entr=[]
    for i in range(5):
        state = gt.minimize_blockmodel_dl(gt_graph,state_args=dict(deg_corr=True))
        for j in range(1000):
            state.multiflip_mcmc_sweep(beta=np.inf, niter=10)
        en=state.entropy()
        st.append(state)
        entr.append(en)
    # minimum
    min_index = entr.index(min(entr))
    # best state
    best_state = st[min_index]
    b = best_state.get_blocks()
    L = {int(v): int(b[v]) for v in gt_graph.vertices()}
    sbm_2[name] = L
    state_2[name] = best_state

In [150]:
# Save the dictionary to a .pkl file
with open("Intermediate_outputs_2/sbm_2.pkl", "wb") as f:
    pickle.dump(sbm_2, f)

# Save the dictionary to a .pkl file
with open("Intermediate_outputs_2/state_2.pkl", "wb") as f:
    pickle.dump(state_2, f)

# Data 3

In [7]:
# Relativity Collab Network
# Load the graph as an undirected network
graph = nx.Graph()

# Read the file and add edges
file_path = "../../Data/ca-GrQc.txt/CA-GrQc.txt"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("#"):  # Skip comment lines
            continue
        node1, node2 = map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()



# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
rel = nx.relabel_nodes(subgraph, mapping)

In [19]:
any(nx.selfloop_edges(rel))

True

In [20]:
self_loops = list(nx.selfloop_edges(rel))
print("Self-loops:", self_loops)

Self-loops: [(210, 210), (1766, 1766), (723, 723), (1763, 1763), (4080, 4080), (2885, 2885)]


In [21]:
# remove self edges
rel.remove_edges_from(nx.selfloop_edges(rel))

In [22]:
any(nx.selfloop_edges(rel))

False

In [8]:
# Lastfm Asia 
file_path = "../../Data/lasftm_asia/lastfm_asia_edges.csv"
df = pd.read_csv(file_path)
# Assuming the file has two columns: 'Source' and 'Target' (adjust if different)
graph = nx.Graph()
edges = list(zip(df.iloc[:, 0], df.iloc[:, 1]))  # Extract edges as tuples
graph.add_edges_from(edges)  # Add edges to the graph

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
lastfm_asia = nx.relabel_nodes(subgraph, mapping)

In [24]:
any(nx.selfloop_edges(lastfm_asia))

False

In [9]:
# Oregon network
graph = nx.Graph()
file_path="../../Data/Oregon_graph/oregon1_010331.txt"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("#"):  # Skip comment lines
            continue
        node1, node2 = map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
oregon = nx.relabel_nodes(subgraph, mapping)

In [26]:
any(nx.selfloop_edges(oregon))

False

In [124]:
data_3= {}
data_3["Relativity collab"] = rel
data_3["Oregon route-views"] = oregon
data_3["Lastfm Asia"]=lastfm_asia

# Save the dictionary to a .pkl file
with open("Intermediate_outputs_3/data_3.pkl", "wb") as f:
    pickle.dump(data_3, f)

In [106]:
# opening files
#with open("Intermediate_outputs_3/data_3.pkl", "rb") as f:
    #new_dict = pickle.load(f)


In [125]:
# create graph tool from nx
# Dictionary to store graph-tool versions of the networks
dict_gt_3 = {}
for name, nx_graph in data_3.items():
    g = gt.Graph(directed=False) 
    # Ensure vertex index matches NetworkX node labels
    g.add_edge_list([(u, v) for u, v in nx_graph.edges()]) 
    dict_gt_3[name] = g

In [127]:
# now create the states of each of them, with minimize sbm
state_3={}
sbm_3 = {}
for name, gt_graph in dict_gt_3.items():
    st=[]
    entr=[]
    for i in range(5):
        state = gt.minimize_blockmodel_dl(gt_graph,state_args=dict(deg_corr=True))
        for j in range(1000):
            state.multiflip_mcmc_sweep(beta=np.inf, niter=10)
        en=state.entropy()
        st.append(state)
        entr.append(en)
    # minimum
    min_index = entr.index(min(entr))
    # best state
    best_state = st[min_index]
    b = best_state.get_blocks()
    L = {int(v): int(b[v]) for v in gt_graph.vertices()}
    sbm_3[name] = L
    state_3[name] = best_state
    print("ok")

ok
ok
ok


In [128]:
# Save the dictionary to a .pkl file
with open("Intermediate_outputs_3/sbm_3.pkl", "wb") as f:
    pickle.dump(sbm_3, f)

# Save the dictionary to a .pkl file
with open("Intermediate_outputs_3/state_3.pkl", "wb") as f:
    pickle.dump(state_3, f)

# Data 4

In [10]:
# Animals Network
# Turtle network
graph = nx.Graph()
file_path="../../Data/reptilia-tortoise-network-fi/reptilia-tortoise-network-fi.edges"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2, date = map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
tortoise_net  = nx.relabel_nodes(subgraph, mapping)


False


In [11]:
# Animals Network
# Weaver network
graph = nx.Graph()
file_path="../../Data/aves-weaver-social/aves-weaver-social.edges"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2, date = map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
weaver_net  = nx.relabel_nodes(subgraph, mapping)

True


In [12]:
# Biological Network
# grid plant
graph = nx.Graph()
file_path="../../Data/bio-grid-plant/bio-grid-plant.edges"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2 = map(int, line.strip().split(","))  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
grid_plant_net  = nx.relabel_nodes(subgraph, mapping)

False


In [13]:
# Biological Network
# grid worm
graph = nx.Graph()
file_path="../../Data/bio-grid-worm/bio-grid-worm.edges"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2 = map(int, line.strip().split(","))  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
grid_worm_net  = nx.relabel_nodes(subgraph, mapping)

False


In [14]:
# Bio
# yeast
# Ho modificato il file mettendo una % all'inizzio della seconda riga, che tanto è numero nodi ed edges
graph = nx.Graph()
file_path="../../Data/bio-yeast/bio-yeast.mtx"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
bio_yeast_net= nx.relabel_nodes(subgraph, mapping)

False


In [15]:
# Power
# US Grid
# anche qui aggiunto % alla riga con quei numeri lì tanto è numero nodi ed edges
graph = nx.Graph()
file_path="../../Data/power-US-Grid/power-US-Grid.mtx"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
power_US_grid= nx.relabel_nodes(subgraph, mapping)

False


In [16]:
# Power
# Eris
# anche qui aggiunto % alla riga con quei numeri lì tanto è numero nodi ed edges
graph = nx.Graph()
file_path="../../Data/power-eris1176/power-eris1176.mtx"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
power_eris= nx.relabel_nodes(subgraph, mapping)

True


In [17]:
# Tech
# PGP
# anche qui aggiunto % alla riga con quei numeri lì tanto è numero nodi ed edges
graph = nx.Graph()
file_path="../../Data/tech-pgp/tech-pgp.edges"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
tech_pgp= nx.relabel_nodes(subgraph, mapping)

False


In [ ]:
data_4 = {}

#data_4["Power: 662 Bus"] = power_662
#data_4["Power: 685 Bus"] = power_685
data_4["Power: US Grid"] = power_US_grid
data_4["Power: Eris"] = power_eris
data_4["Animal net: Weaver"] = weaver_net 
data_4["Animal net: Tortoise"] = tortoise_net
data_4["Bio: plant"] = grid_plant_net 
data_4["Bio: worm"] = grid_worm_net 
data_4["Bio: yeast"] = bio_yeast_net
data_4["Tech: PGP"] = tech_pgp

# Save the dictionary to a .pkl file
with open("Intermediate_outputs_4/data_4.pkl", "wb") as f:
    pickle.dump(data_4, f)

In [45]:
# create graph tool from nx
# Dictionary to store graph-tool versions of the networks
dict_gt_4 = {}
for name, nx_graph in data_4.items():
    g = gt.Graph(directed=False) 
    # Ensure vertex index matches NetworkX node labels
    g.add_edge_list([(u, v) for u, v in nx_graph.edges()]) 
    dict_gt_4[name] = g

In [46]:
# now create the states of each of them, with minimize sbm
state_4={}
sbm_4= {}
for name, gt_graph in dict_gt_4.items():
    st=[]
    entr=[]
    for i in range(5):
        state = gt.minimize_blockmodel_dl(gt_graph,state_args=dict(deg_corr=True))
        for j in range(1000):
            state.multiflip_mcmc_sweep(beta=np.inf, niter=10)
        en=state.entropy()
        st.append(state)
        entr.append(en)
    # minimum
    min_index = entr.index(min(entr))
    # best state
    best_state = st[min_index]
    b = best_state.get_blocks()
    L = {int(v): int(b[v]) for v in gt_graph.vertices()}
    sbm_4[name] = L
    state_4[name] = best_state
    print("ok")

ok
ok
ok
ok
ok
ok
ok
ok


In [47]:
# Save the dictionary to a .pkl file
with open("Intermediate_outputs_4/sbm_4.pkl", "wb") as f:
    pickle.dump(sbm_4, f)

# Save the dictionary to a .pkl file
with open("Intermediate_outputs_4/state_4.pkl", "wb") as f:
    pickle.dump(state_4, f)

# Data 5 

In [18]:
# Infrastructure
# anche qui aggiunto % alla riga con quei numeri lì tanto è numero nodi ed edges
graph = nx.Graph()
file_path="../../Data/inf-openflights/inf-openflights.edges"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
inf_open_flights= nx.relabel_nodes(subgraph, mapping)

False


In [19]:
# Infrastructure
graph = nx.Graph()
file_path="../../Data/power-bcspwr09/power-bcspwr09.mtx"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
bcs_09= nx.relabel_nodes(subgraph, mapping)

True


In [20]:
# Power
graph = nx.Graph()
file_path="../../Data/power-bcspwr10/power-bcspwr10.mtx"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
bcs_10= nx.relabel_nodes(subgraph, mapping)

True


In [21]:
# Road
graph = nx.Graph()
file_path="../../Data/road-minnesota/road-minnesota.mtx"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
road_M= nx.relabel_nodes(subgraph, mapping)

False


In [22]:
# Tech route
graph = nx.Graph()
file_path="../../Data/tech-routers-rf/tech-routers-rf.mtx"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
tech_rf= nx.relabel_nodes(subgraph, mapping)

False


In [23]:
# tech WHOIS
graph = nx.Graph()
file_path="../../Data/tech-WHOIS/tech-WHOIS.mtx"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
tech_whois= nx.relabel_nodes(subgraph, mapping)

False


In [24]:
# Web Edu
graph = nx.Graph()
file_path="../../Data/web-edu/web-edu.mtx"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
web_edu= nx.relabel_nodes(subgraph, mapping)

False


In [25]:
# Web polblogs
graph = nx.Graph()
file_path="../../Data/web-polblogs/web-polblogs.mtx"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
web_polblogs= nx.relabel_nodes(subgraph, mapping)

False


In [93]:
# Saving results
data_5 = {}
data_5["Flights"] = inf_open_flights
data_5["Power: bcspwr09"] = bcs_09
data_5["Power: bcspwr10"] = bcs_10
data_5["Minnesota Road"] = road_M
data_5["Tech: Routers rf"] = tech_rf
data_5["Tech: WHOIS"] = tech_whois
data_5["Web: edu"] = web_edu
data_5["Web: polblogs"] = web_polblogs

# Save the dictionary to a .pkl file
with open("Intermediate_outputs_5/data_5.pkl", "wb") as f:
    pickle.dump(data_5, f)

In [125]:
# create graph tool from nx
# Dictionary to store graph-tool versions of the networks
dict_gt_5 = {}
for name, nx_graph in data_5.items():
    g = gt.Graph(directed=False) 
    # Ensure vertex index matches NetworkX node labels
    g.add_edge_list([(u, v) for u, v in nx_graph.edges()]) 
    dict_gt_5[name] = g

In [95]:
# now create the states of each of them, with minimize sbm
state_5={}
sbm_5= {}
for name, gt_graph in dict_gt_5.items():
    st=[]
    entr=[]
    for i in range(5):
        state = gt.minimize_blockmodel_dl(gt_graph,state_args=dict(deg_corr=True))
        for j in range(1000):
            state.multiflip_mcmc_sweep(beta=np.inf, niter=10)
        en=state.entropy()
        st.append(state)
        entr.append(en)
    # minimum
    min_index = entr.index(min(entr))
    # best state
    best_state = st[min_index]
    b = best_state.get_blocks()
    L = {int(v): int(b[v]) for v in gt_graph.vertices()}
    sbm_5[name] = L
    state_5[name] = best_state
    print("ok")

ok
ok
ok
ok
ok
ok
ok
ok


In [96]:
# Save the dictionary to a .pkl file
with open("Intermediate_outputs_5/sbm_5.pkl", "wb") as f:
    pickle.dump(sbm_5, f)

# Save the dictionary to a .pkl file
with open("Intermediate_outputs_5/state_5.pkl", "wb") as f:
    pickle.dump(state_5, f)

# Data 6

In [26]:
# Collab CS
graph = nx.Graph()
file_path="../../Data/ca-CSphd/ca-CSphd.mtx"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
ca_cs= nx.relabel_nodes(subgraph, mapping)

False


In [27]:
# Collab Erdos
graph = nx.Graph()
file_path="../../Data/ca-Erdos992/ca-Erdos992.mtx"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
ca_erdos= nx.relabel_nodes(subgraph, mapping)

False


In [28]:
# social anybeat
graph = nx.Graph()
file_path="../../Data/soc-anybeat/soc-anybeat.edges"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
anybeat = nx.relabel_nodes(subgraph, mapping)

False


In [29]:
# FB American75
graph = nx.Graph()
file_path="../../Data/socfb-American75/socfb-American75.mtx"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
a75= nx.relabel_nodes(subgraph, mapping)

False


In [30]:
# FB Colgate88
graph = nx.Graph()
file_path="../../Data/socfb-Colgate88/socfb-Colgate88.mtx"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
colgate88= nx.relabel_nodes(subgraph, mapping)

False


In [31]:
# FB Haverford
graph = nx.Graph()
file_path="../../Data/socfb-Haverford76/socfb-Haverford76.mtx"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
h76= nx.relabel_nodes(subgraph, mapping)

False


In [32]:
# FB UC64
graph = nx.Graph()
file_path="../../Data/socfb-UC64/socfb-UC64.mtx"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
uc64= nx.relabel_nodes(subgraph, mapping)

False


In [33]:
# FB USFCA
graph = nx.Graph()
file_path="../../Data/socfb-USFCA72/socfb-USFCA72.mtx"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
usfca72= nx.relabel_nodes(subgraph, mapping)

False


In [34]:
# social Hamsterster
graph = nx.Graph()
file_path="../../Data/soc-hamsterster/soc-hamsterster.edges"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
ham= nx.relabel_nodes(subgraph, mapping)

False


In [35]:
# FB gov
graph = nx.Graph()
file_path="../../Data/fb-pages-government/fb-pages-government.edges"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.strip().split(",")) # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
fb_gov= nx.relabel_nodes(subgraph, mapping)

True


In [14]:
# Saving results
data_6 = {}
data_6["FB: Gov."] = fb_gov
data_6["Hamsterster"] = ham
data_6["FB: USFCA72"] = usfca72
data_6["FB: UC64"] = uc64
data_6["FB: Haverford76"] = h76
data_6["FB: Colgate88"] = colgate88
data_6["FB: American75"] = a75
data_6["Anybeat"] = anybeat
data_6["Erdos Collab"] = ca_erdos
data_6["CS Collab"] = ca_cs

# Save the dictionary to a .pkl file
with open("Intermediate_outputs_6/data_6.pkl", "wb") as f:
    pickle.dump(data_6, f)

In [15]:
# create graph tool from nx
# Dictionary to store graph-tool versions of the networks
dict_gt_6 = {}
for name, nx_graph in data_6.items():
    g = gt.Graph(directed=False) 
    # Ensure vertex index matches NetworkX node labels
    g.add_edge_list([(u, v) for u, v in nx_graph.edges()]) 
    dict_gt_6[name] = g

In [16]:
# now create the states of each of them, with minimize sbm
state_6={}
sbm_6= {}
for name, gt_graph in dict_gt_6.items():
    st=[]
    entr=[]
    for i in range(5):
        state = gt.minimize_blockmodel_dl(gt_graph,state_args=dict(deg_corr=True))
        for j in range(1000):
            state.multiflip_mcmc_sweep(beta=np.inf, niter=10)
        en=state.entropy()
        st.append(state)
        entr.append(en)
    # minimum
    min_index = entr.index(min(entr))
    # best state
    best_state = st[min_index]
    b = best_state.get_blocks()
    L = {int(v): int(b[v]) for v in gt_graph.vertices()}
    sbm_6[name] = L
    state_6[name] = best_state
    print("ok")

ok
ok
ok
ok
ok
ok
ok
ok
ok
ok


In [17]:
# Save the dictionary to a .pkl file
with open("Intermediate_outputs_6/sbm_6.pkl", "wb") as f:
    pickle.dump(sbm_6, f)

# Save the dictionary to a .pkl file
with open("Intermediate_outputs_6/state_6.pkl", "wb") as f:
    pickle.dump(state_6, f)

# Data 7

In [36]:
# Web Indochina
graph = nx.Graph()
file_path="../../Data/web-indochina-2004/web-indochina-2004.mtx"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
web_indoc= nx.relabel_nodes(subgraph, mapping)

False


In [37]:
# Web EPA
graph = nx.Graph()
file_path="../../Data/web-EPA/web-EPA.edges"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
web_epa= nx.relabel_nodes(subgraph, mapping)

False


In [38]:
# Web
graph = nx.Graph()
file_path="../../Data/web-spam/web-spam.mtx"
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("%"):  # Skip comment lines
            continue
        node1, node2= map(int, line.split())  # Read edge
        graph.add_edge(node1, node2)  # Add as undirected

print(any(nx.selfloop_edges(graph)))
graph.remove_edges_from(nx.selfloop_edges(graph))

# Extract the largest connected component
largest_cc = max(nx.connected_components(graph), key=len)
subgraph = graph.subgraph(largest_cc).copy()

# Relabel nodes in ascending order
mapping = {node: idx for idx, node in enumerate(sorted(subgraph.nodes()))}
web_spam= nx.relabel_nodes(subgraph, mapping)

False


In [138]:
# Saving results
data_7 = {}

data_7["Web: Indochina"] = web_indoc
data_7["Web: EPA"] = web_epa
data_7["Web: spam"] = web_spam

# Save the dictionary to a .pkl file
with open("Intermediate_outputs_7/data_7.pkl", "wb") as f:
    pickle.dump(data_7, f)

In [139]:
# create graph tool from nx
# Dictionary to store graph-tool versions of the networks
dict_gt_7 = {}
for name, nx_graph in data_7.items():
    g = gt.Graph(directed=False) 
    # Ensure vertex index matches NetworkX node labels
    g.add_edge_list([(u, v) for u, v in nx_graph.edges()]) 
    dict_gt_7[name] = g

In [140]:
# now create the states of each of them, with minimize sbm
state_7={}
sbm_7= {}
for name, gt_graph in dict_gt_7.items():
    st=[]
    entr=[]
    for i in range(5):
        state = gt.minimize_blockmodel_dl(gt_graph,state_args=dict(deg_corr=True))
        for j in range(1000):
            state.multiflip_mcmc_sweep(beta=np.inf, niter=10)
        en=state.entropy()
        st.append(state)
        entr.append(en)
    # minimum
    min_index = entr.index(min(entr))
    # best state
    best_state = st[min_index]
    b = best_state.get_blocks()
    L = {int(v): int(b[v]) for v in gt_graph.vertices()}
    sbm_7[name] = L
    state_7[name] = best_state
    print("ok")

ok
ok
ok


In [141]:
# Save the dictionary to a .pkl file
with open("Intermediate_outputs_7/sbm_7.pkl", "wb") as f:
    pickle.dump(sbm_7, f)

# Save the dictionary to a .pkl file
with open("Intermediate_outputs_7/state_7.pkl", "wb") as f:
    pickle.dump(state_7, f)